# Sickling — Colab demo

End-to-end demo of the two ML models from Goreke et al. (Mol Ther Adv), applied to a single bright-field FOV:

1. **U-Net HbS-protrusion segmentation** — 4-class pixel mask (protrusion / background / cell body / cell boundary).
2. **Watershed instance segmentation** — one integer label per RBC, from the U-Net mask.
3. **DINOv2 + morphology sickle classifier** — sickle vs non-sickle per cell.
4. **Per-FOV summary** — total HbS-protrusion length (µm), sickle fraction, µm protrusion per sickle cell.

**Runtime:** ~2 min on a Colab T4 GPU. Sample image bundled in the repo (`sample.jpg`). Model checkpoints are on Google Drive — paste the two `drive.google.com` URLs into the *Setup* cell before running (see the repo `README.md` for links).

## 1. Setup — install the package, fetch sample + checkpoints

In [ ]:
# Install the sickling package from GitHub + gdown for Google Drive downloads.
REPO = 'github.com/ugoreke/sickling'
%pip install -q git+https://{REPO}.git gdown

# ─── Google Drive checkpoint URLs ───────────────────────────────────────
# Standard Drive share links of the form
#   https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing
# work directly — gdown extracts <FILE_ID> automatically.
UNET_CHECKPOINT_URL = 'https://drive.google.com/file/d/1q9nCoRHeqFkuMzcyANWAHHhREUsTapr9/view?usp=sharing'  # 4-class U-Net (~118 MB)
CLF_CHECKPOINT_URL  = 'https://drive.google.com/file/d/1Q0qLOU6xy9TCS4Hqhhd33dN29010R3Ya/view?usp=sharing'  # DINOv2 + morphology classifier (~87 MB)

# The sample image is small enough to ship in the repo; grab it via raw GitHub.
SAMPLE_IMG_URL = f'https://raw.githubusercontent.com/{REPO.split("github.com/")[-1]}/main/sample.jpg'

import urllib.request
import gdown
from pathlib import Path
DEMO_DIR = Path('demo_data'); DEMO_DIR.mkdir(exist_ok=True)
UNET_PATH   = DEMO_DIR / 'unet.pth'
CLF_PATH    = DEMO_DIR / 'classifier.ckpt'
SAMPLE_PATH = DEMO_DIR / 'sample.jpg'

def _gdown(url, dest):
    if not dest.exists():
        # fuzzy=True → extract the file id from any of the standard Drive URL shapes.
        gdown.download(url, str(dest), quiet=False, fuzzy=True)
    print(f'  ✓ {dest.name} ({dest.stat().st_size/1e6:.1f} MB)')

def _urlget(url, dest):
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    print(f'  ✓ {dest.name} ({dest.stat().st_size/1e6:.1f} MB)')

_urlget(SAMPLE_IMG_URL,     SAMPLE_PATH)
_gdown( UNET_CHECKPOINT_URL, UNET_PATH)
_gdown( CLF_CHECKPOINT_URL,  CLF_PATH)

## 2. Imports + sample image

In [ ]:
import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image

mpl.rcParams['svg.fonttype'] = 'none'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

raw = np.array(Image.open(SAMPLE_PATH).convert('L'))
print(f'sample image: {raw.shape}  dtype={raw.dtype}')

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(raw, cmap='gray')
ax.set_title('sample.jpg — raw bright-field')
ax.axis('off')
plt.show()

## 3. U-Net protrusion segmentation

Loads the 4-class U-Net checkpoint and produces a per-pixel argmax mask. Class indices: `0 = HbS protrusion`, `1 = background`, `2 = cell body`, `3 = cell boundary`.

In [ ]:
from sickling.rbc_classification.stage1_unet.inference import (
    load_unet, predict_label_map,
)
from sickling.rbc_classification.io.images import normalize_image

unet = load_unet(UNET_PATH, n_classes=4, device=DEVICE)
raw_norm = normalize_image(raw, percentile=99.0)
label_map = predict_label_map(unet, raw_norm, tile_size=256, overlap=0.5)
print(f'label_map: {label_map.shape}  classes={np.unique(label_map).tolist()}')

# Render the 4-class mask alongside the raw image.
PALETTE = np.array([
    [200,  40,  40],   # 0 protrusion — red
    [255, 255, 255],   # 1 background — white
    [180, 180, 200],   # 2 cell body — light gray-blue
    [ 30,  30,  30],   # 3 cell boundary — near black
], dtype=np.uint8)
overlay = PALETTE[label_map]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(raw, cmap='gray'); axes[0].set_title('raw'); axes[0].axis('off')
axes[1].imshow(overlay); axes[1].set_title('U-Net 4-class mask'); axes[1].axis('off')
plt.tight_layout(); plt.show()

# Per-class pixel fractions — quick sanity
for c, name in enumerate(['protrusion', 'background', 'cell_body', 'cell_boundary']):
    frac = (label_map == c).mean()
    print(f'  class {c} ({name:14s}): {frac*100:5.2f} %')

## 4. Quantify protrusion — connected components + major-axis length

In [ ]:
from skimage.measure import label as sklabel
from skimage.measure import regionprops

# Calibration: Leica 40× — 500 µm per 3100 px
PX_TO_UM = 500.0 / 3100.0
MIN_LENGTH_PX = 10

protrusion_mask = (label_map == 0).astype(np.uint8)
cc = sklabel(protrusion_mask, connectivity=2)
props = regionprops(cc)

kept_lengths_um, dropped = [], 0
for p in props:
    if p.major_axis_length < MIN_LENGTH_PX:
        dropped += 1
        continue
    kept_lengths_um.append(p.major_axis_length * PX_TO_UM)
kept_lengths_um = np.array(kept_lengths_um)

print(f'{len(props)} components detected,  {len(kept_lengths_um)} kept,  {dropped} dropped (<{MIN_LENGTH_PX} px)')
print(f'total protrusion length : {kept_lengths_um.sum():>7.2f} µm')
print(f'mean component length   : {kept_lengths_um.mean():>7.2f} µm') if len(kept_lengths_um) else None

## 5. Watershed instance segmentation

Foreground = cell_body ∪ protrusion. Morphological closing → EDT → marker-seeded watershed gives one integer label per RBC.

In [ ]:
from sickling.rbc_classification.stage2_instances.watershed import mask_to_instances

instance_map = mask_to_instances(label_map)
n_instances = int(instance_map.max())
print(f'instances detected: {n_instances}')

# Visualize
from matplotlib.colors import ListedColormap
rng = np.random.default_rng(0)
colors = rng.uniform(0.3, 1.0, size=(n_instances + 1, 3))
colors[0] = (0, 0, 0)   # background → black
cmap = ListedColormap(colors)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(raw, cmap='gray'); axes[0].set_title('raw'); axes[0].axis('off')
axes[1].imshow(instance_map, cmap=cmap, interpolation='nearest')
axes[1].set_title(f'{n_instances} watershed instances'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 6. Sickle / non-sickle classification

For each watershed instance we extract a 96×96×3 crop (channel 0 = raw, ch 1 = cell-body mask, ch 2 = protrusion mask), compute 30 hand-crafted morphology features, and run the multimodal classifier.

In [ ]:
from sickling.rbc_classification.stage3_crops.extract import extract_for_fov
from sickling.rbc_classification.stage5_multimodal.morphology_features import compute_features, FEATURE_NAMES
from sickling.rbc_classification.stage4_repr import build_encoder
from sickling.rbc_classification.stage5_multimodal.classifier import MultimodalClassifier
from sickling.rbc_classification.stage5_multimodal.image_tower import ImageTower
from sickling.rbc_classification.stage5_multimodal.morphology_tower import MorphologyTower
from sickling.rbc_classification.stage5_multimodal.lightning_module import MultimodalFinetuneModule
from sickling.rbc_classification.stage5_multimodal.morphology_features import N_FEATURES

# Per-FOV crop + per-cell morphology.
crops, morph_rows, instance_ids = extract_for_fov(
    raw=raw_norm, label_map=label_map, instance_map=instance_map,
)
print(f'extracted {len(crops)} cell crops of shape {crops[0].shape if len(crops) else None}')

# Build the multimodal classifier (DINOv2 frozen + morphology tower + 2-layer fusion).
encoder = build_encoder('dinov2_frozen')
image_tower = ImageTower(encoder)
morph_tower = MorphologyTower(in_features=N_FEATURES)
classifier = MultimodalClassifier(
    towers={'image': image_tower, 'morphology': morph_tower},
    num_classes=2, hidden=256, dropout=0.2,
)
module = MultimodalFinetuneModule(classifier=classifier)
state = torch.load(CLF_PATH, map_location='cpu', weights_only=False)
module.load_state_dict(state['state_dict'], strict=False)
module = module.to(DEVICE).eval()

# Run inference.
with torch.no_grad():
    crops_t  = torch.from_numpy(np.stack(crops)).float().to(DEVICE)
    morphs_t = torch.from_numpy(np.stack(morph_rows)).float().to(DEVICE)
    logits   = module.classifier({'image': crops_t, 'morphology': morphs_t})
    p_sickle = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

THRESHOLD = 0.6   # matches the OOF MCC-max threshold from the supplementary figure
y_pred = (p_sickle >= THRESHOLD).astype(np.int64)
n_sickle = int(y_pred.sum()); n_total = int(y_pred.size)
print(f'classified {n_total} cells:  {n_sickle} sickle  ({n_sickle/n_total*100:.1f}%)')

## 7. Final overlay — cells colored by predicted label

In [ ]:
# Map each instance_id to its predicted class.
label_per_instance = {iid: int(yp) for iid, yp in zip(instance_ids, y_pred)}
overlay_pred = np.zeros((*instance_map.shape, 3), dtype=np.uint8)
for iid, lab in label_per_instance.items():
    mask = instance_map == iid
    overlay_pred[mask] = (200, 40, 40) if lab == 1 else (60, 130, 200)
# Background: gray
overlay_pred[instance_map == 0] = (40, 40, 40)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(raw, cmap='gray'); axes[0].set_title('raw'); axes[0].axis('off')
axes[1].imshow(overlay_pred)
axes[1].set_title(f'cell labels  —  red = sickle ({n_sickle}),  blue = non-sickle ({n_total-n_sickle})')
axes[1].axis('off')
plt.tight_layout(); plt.show()

# Per-FOV summary the paper reports.
polymer_length_um = float(kept_lengths_um.sum())
sickle_fraction = n_sickle / n_total if n_total else float('nan')
um_per_sickle = polymer_length_um / n_sickle if n_sickle else float('nan')
print()
print('=' * 50)
print('  PER-FOV SUMMARY (this single sample image)')
print('=' * 50)
print(f'  total HbS protrusion length : {polymer_length_um:>7.2f} µm')
print(f'  cells detected              : {n_total:>4d}')
print(f'  sickle cells                : {n_sickle:>4d}  ({sickle_fraction*100:.1f} %)')
print(f'  µm protrusion / sickle cell : {um_per_sickle:>7.2f}')

## Where to go next

- **How the classifier was evaluated:** `notebooks/sickle_classifier_confusion_matrix.ipynb` regenerates the PR + confusion matrix + calibration triptych straight from the 5 committed fold OOF prediction dumps — no checkpoints needed.
- **How the U-Net was evaluated at the pixel level:** `notebooks/pixel_confusion_matrix.ipynb` computes 4-class recall / precision / F1 on the held-out `InitialLabels` FOVs.
- **How the U-Net protrusion length was validated:** `notebooks/protrusion_length_grid.ipynb` compares U-Net-derived protrusion length to manual Photoshop-Count measurements on a 10×10 evaluation grid.
- **How the biology metric is computed:** `notebooks/analysis_protrusion_per_condition.ipynb` shows the full pipeline from per-FOV parquets to per-condition pool statistics with bootstrap CIs.
- **How each model is trained:** `sickling/protrusion_detection/HITL_pipeline.ipynb` (U-Net HITL loop) and `sickling/rbc_classification/orchestrate.ipynb` (classifier).